# Process Dataset: the 30-patient test set

This notebook builds the test files used in the revision from `data/test_rtf_v2.csv`, which holds the 30 real hold-out patients (10 poor prognosis, 20 good prognosis) with their sample IDs. The 10 SMOTE rows of the submitted 40-row test set are not in it.

| Step | Input | Output |
|---|---|---|
| 1 | `test_rtf_v2.csv` | `test_rtf_v2_cleaned.csv`: original scale, without the ID and text-class columns |
| 2 | `rtfDataSet.csv` | min-max parameters (minimum and maximum of each gene) |
| 3 | `test_rtf_v2_cleaned.csv` and the step 2 parameters | `test_rtf_v2_cleaned_normal.csv` |
| 4 | the files above, `test_normal_rtf.csv`, `split_raw_ids.csv` | checks only, nothing written |

**Which min-max parameters, and why.** The classifiers are trained on `train_normal_rtf.csv`. That file was produced in 2023 in this order:

1. WEKA SMOTE (k = 5, 100%) over all 295 patients, giving `rtfDataSet.csv` (396 rows).
2. Min-max fitted on those 396 rows, giving `normal_rtf.csv`.
3. A random 90/10 split of the 396 rows into 356 training rows and 40 test rows.

The training file was not re-scaled for the revision. The test patients must go through the same transformation as the training file, so step 2 takes the minimum and maximum of each gene over the 396 rows of `rtfDataSet.csv`. Each SMOTE row lies between two real patients, so this range is the range of the 295 real patients, including these 30 test patients (checked in step 2). **These are not training-only parameters.**

As a consequence, the 30 patients keep exactly the normalized values they had in the submitted `test_normal_rtf.csv`. Re-normalizing them gives the same result as deleting the 10 SMOTE rows (checked in step 4).

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = '../data/'
LABEL = 'label'

## 1. Remove the ID and text-class columns

`test_rtf_v2.csv` has `SampleID`, the 70 genes `0`–`69`, `label` and `Class(Obs)` (the label as text). The analysis files keep only the genes and `label`. The row order is unchanged, so the sample ID of row *i* is in row *i* of `test_rtf_v2.csv`.

In [2]:
test_ids = pd.read_csv(DATA_DIR + 'test_rtf_v2.csv')
test_raw = test_ids.drop(['SampleID', 'Class(Obs)'], axis=1)
test_raw.to_csv(DATA_DIR + 'test_rtf_v2_cleaned.csv', index=False)

GENES = [c for c in test_raw.columns if c != LABEL]
assert len(GENES) == 70
print(f'test_rtf_v2_cleaned.csv: {len(test_raw)} patients, '
      f'labels {test_raw[LABEL].value_counts().sort_index().to_dict()}')

test_rtf_v2_cleaned.csv: 30 patients, labels {0: 20, 1: 10}


## 2. Min-max parameters

These are the minimum and maximum of each gene over the 396 rows of `rtfDataSet.csv`, the same parameters that produced `normal_rtf.csv` and `train_normal_rtf.csv`.

In `rtfDataSet.csv`, rows 0–294 are the 295 real patients (values with 3 decimals) and rows 295–395 are the 101 SMOTE rows (all positive, more decimals). The cell checks three things:

- the 396-row range equals the range of the 295 real patients;
- `normal_rtf.csv` and `train_normal_rtf.csv` are these parameters applied to their raw rows;
- the training rows alone would give different parameters: in `train_normal_rtf.csv` some genes never reach 0 or 1, because their extremes are in test rows.

In [3]:
reference = pd.read_csv(DATA_DIR + 'rtfDataSet.csv')
assert list(reference.columns) == GENES + [LABEL] and len(reference) == 396

gene_min = reference[GENES].min()
gene_max = reference[GENES].max()

def min_max(df):
    # (x - min) / (max - min) with the parameters above. Written as a formula rather than
    # sklearn's MinMaxScaler, which computes x * scale + offset and can differ in the last
    # floating-point digit; this form reproduces the committed files byte for byte.
    return (df[GENES] - gene_min) / (gene_max - gene_min)

# Rows 0-294: the 295 real patients (3 decimals). Rows 295-395: the 101 SMOTE rows.
real, synthetic = reference.iloc[:295], reference.iloc[295:]
three_decimals = lambda df: np.allclose(df[GENES] * 1000, (df[GENES] * 1000).round(), atol=1e-6)
assert three_decimals(real) and not three_decimals(synthetic)
assert real[LABEL].value_counts().to_dict() == {0: 194, 1: 101}
assert (synthetic[LABEL] == 1).all()
assert gene_min.equals(real[GENES].min()) and gene_max.equals(real[GENES].max())
print('396-row range == range of the 295 real patients (including the 30 test patients)')

# normal_rtf.csv and train_normal_rtf.csv were scaled with these parameters.
normal = pd.read_csv(DATA_DIR + 'normal_rtf.csv')
train_raw = pd.read_csv(DATA_DIR + 'train_rtf.csv')
train_norm = pd.read_csv(DATA_DIR + 'train_normal_rtf.csv')
diff_normal = (min_max(reference) - normal[GENES]).abs().to_numpy().max()
diff_train = (min_max(train_raw) - train_norm[GENES]).abs().to_numpy().max()
assert diff_normal < 1e-12 and diff_train < 1e-12
print(f'normal_rtf.csv       = min-max(rtfDataSet.csv): max |diff| = {diff_normal:.1e}')
print(f'train_normal_rtf.csv = min-max(train_rtf.csv):  max |diff| = {diff_train:.1e}')

# Parameters fitted on the training rows would put every gene's minimum at 0 and maximum at 1.
not_zero = int((train_norm[GENES].min() > 1e-12).sum())
not_one = int((train_norm[GENES].max() < 1 - 1e-12).sum())
assert not_zero > 0 and not_one > 0
print(f'In train_normal_rtf.csv, {not_zero} genes never reach 0 and {not_one} never reach 1: '
      'the scaling was not fitted on the training rows')

396-row range == range of the 295 real patients (including the 30 test patients)
normal_rtf.csv       = min-max(rtfDataSet.csv): max |diff| = 2.2e-16
train_normal_rtf.csv = min-max(train_rtf.csv):  max |diff| = 3.3e-16
In train_normal_rtf.csv, 9 genes never reach 0 and 12 never reach 1: the scaling was not fitted on the training rows


## 3. Normalize the test patients

In [4]:
test_norm = min_max(test_raw)
test_norm[LABEL] = test_raw[LABEL]
test_norm.to_csv(DATA_DIR + 'test_rtf_v2_cleaned_normal.csv', index=False)
print(f'test_rtf_v2_cleaned_normal.csv: {test_norm.shape[0]} patients, {len(GENES)} genes + label')

test_rtf_v2_cleaned_normal.csv: 30 patients, 70 genes + label


## 4. Checks

- All test values are in [0, 1], because the 30 patients are part of the reference range.
- Each of the 30 rows equals one of the 40 rows of the submitted `test_normal_rtf.csv`, with the same label.
- The 10 rows of `test_normal_rtf.csv` left unmatched are exactly its SMOTE rows, i.e. the test rows whose `raw_id` in `split_raw_ids.csv` is 295 or more.

In [5]:
assert test_norm[GENES].to_numpy().min() >= 0 and test_norm[GENES].to_numpy().max() <= 1

old_test = pd.read_csv(DATA_DIR + 'test_normal_rtf.csv')
dist = np.abs(test_norm[GENES].to_numpy()[:, None, :] - old_test[GENES].to_numpy()[None, :, :]).max(axis=2)
match = dist.argmin(axis=1)
assert dist.min(axis=1).max() < 1e-12 and len(set(match)) == 30
assert (old_test[LABEL].to_numpy()[match] == test_norm[LABEL].to_numpy()).all()

ids = pd.read_csv(DATA_DIR + 'split_raw_ids.csv')
old_smote_rows = set(ids.query("split == 'test' and raw_id >= 295")['split_row'])
assert set(range(len(old_test))) - set(match) == old_smote_rows
print(f'The 30 patients have the same normalized values as in test_normal_rtf.csv '
      f'(max |diff| = {dist.min(axis=1).max():.1e}); the {len(old_smote_rows)} rows left out are its SMOTE rows')

The 30 patients have the same normalized values as in test_normal_rtf.csv (max |diff| = 3.3e-16); the 10 rows left out are its SMOTE rows


## Output files

Both files have one row per patient, in the row order of `test_rtf_v2.csv` (the sample IDs are there and in `test_rtf_with_ID_original.csv`).

| Column | `test_rtf_v2_cleaned.csv` | `test_rtf_v2_cleaned_normal.csv` |
|---|---|---|
| `0` … `69` | Expression of the 70 MammaPrint genes on the original scale of the data set. Column *k* is gene g*k*+1 of the manuscript. | The same genes after min-max scaling with the step 2 parameters, in [0, 1]. |
| `label` | 1 = poor prognosis (metastatic event), 0 = good prognosis. | Same. |